In [1]:
import cv2
import numpy as np
import pickle
from collections import deque
from tensorflow.keras.models import load_model
from fsl_preprocessing import normalize_landmarks
  

# Load model
model = load_model("models/dynamic/subset_lstm_model.h5", compile=False)

# Load label encoder
with open("models/dynamic/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Load preprocessing config
with open("models/dynamic/preprocess_config.pkl", "rb") as f:
    config = pickle.load(f)

SEQ_LENGTH = config["seq_length"]

sequence = deque(maxlen=SEQ_LENGTH)


In [2]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)


In [3]:
import cv2
import numpy as np
import pickle
from collections import deque
from tensorflow.keras.models import load_model
from fsl_preprocessing import normalize_landmarks
  

# Load model
model = load_model("models/dynamic/subset_lstm_model.h5", compile=False)

# Load label encoder
with open("models/dynamic/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Load preprocessing config
with open("models/dynamic/preprocess_config.pkl", "rb") as f:
    config = pickle.load(f)

SEQ_LENGTH = config["seq_length"]

sequence = deque(maxlen=SEQ_LENGTH)


import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("Camera failed to open")
    exit()


from collections import deque, Counter

sequence = deque(maxlen=SEQ_LENGTH)
predictions = deque(maxlen=10)   # limit smoothing window

CONF_THRESHOLD = 0.5



print("Starting webcam...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    display_label = "..."

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])

        landmarks = normalize_landmarks(landmarks)
        sequence.append(landmarks)

        if len(sequence) == SEQ_LENGTH:
            input_data = np.expand_dims(sequence, axis=0)
            prediction = model.predict(input_data, verbose=0)
            print("Raw prediction:", prediction)

            confidence = np.max(prediction)
            class_id = np.argmax(prediction)

            label = le.inverse_transform([class_id])[0]
            display_label = f"{label} ({confidence:.2f})"


            if confidence > CONF_THRESHOLD:
                predictions.append(class_id)

                # smoothing
                final_class = Counter(predictions).most_common(1)[0][0]
                display_label = le.inverse_transform([final_class])[0]

            print("Confidence:", confidence)
            print("Prediction:", prediction)
            print("Sequence len:", len(sequence))
            print("Confidence:", confidence if len(sequence)==SEQ_LENGTH else "N/A")
    else:
        sequence.clear()
        predictions.clear()

    cv2.putText(frame,
                display_label,
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

    cv2.imshow("Dynamic Sign Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Starting webcam...


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Raw prediction: [[0.31908113 0.35641417 0.32450476]]
Confidence: 0.35641417
Prediction: [[0.31908113 0.35641417 0.32450476]]
Sequence len: 30
Confidence: 0.35641417
Raw prediction: [[0.31842142 0.35762084 0.32395777]]
Confidence: 0.35762084
Prediction: [[0.31842142 0.35762084 0.32395777]]
Sequence len: 30
Confidence: 0.35762084


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Raw prediction: [[0.31790426 0.35836276 0.32373294]]
Confidence: 0.35836276
Prediction: [[0.31790426 0.35836276 0.32373294]]
Sequence len: 30
Confidence: 0.35836276
Raw prediction: [[0.3173948  0.35903198 0.32357332]]
Confidence: 0.35903198
Prediction: [[0.3173948  0.35903198 0.32357332]]
Sequence len: 30
Confidence: 0.35903198


...

Raw prediction: [[0.3162467  0.34983763 0.33391568]]
Confidence: 0.34983763
Prediction: [[0.3162467  0.34983763 0.33391568]]
Sequence len: 30
Confidence: 0.34983763
Raw prediction: [[0.31630945 0.34973875 0.33395186]]
Confidence: 0.34973875
Prediction: [[0.31630945 0.34973875 0.33395186]]
Sequence len: 30
Confidence: 0.34973875


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Raw prediction: [[0.31634432 0.3496705  0.33398515]]
Confidence: 0.3496705
Prediction: [[0.31634432 0.3496705  0.33398515]]
Sequence len: 30
Confidence: 0.3496705
Raw prediction: [[0.31634206 0.34966624 0.33399162]]
Confidence: 0.34966624
Prediction: [[0.31634206 0.34966624 0.33399162]]
Sequence len: 30
Confidence: 0.34966624


c:\Projects\signia-fsl-recognition\fsl39\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

In [ ]:
print("Sample sequence:")
print(sequence[:2])

Sample sequence:


TypeError: sequence index must be integer, not 'slice'